# Семинар 14: Отбор признаков.



In [ ]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

## Данные

Мы будем работать с данными из соревнования Home Credit Default Risk в котором требовалось предсказать вернет ли клиент кредит.

https://www.kaggle.com/c/home-credit-default-risk

In [ ]:
# Либо так:

# https://drive.google.com/file/d/11ru9cmTaGaQs-AyaFAmGmCmJOM6-EHtD/view?usp=sharing

In [ ]:
import gdown

file_id = '11ru9cmTaGaQs-AyaFAmGmCmJOM6-EHtD'
output_name = 'application_train.csv'

gdown.download(id=file_id, output=output_name, quiet=False, fuzzy=True, use_cookies=False)

Загрузим данные и посмотрим на них.

In [ ]:
application_train = pd.read_csv(output_name)
application_train.shape

In [ ]:
application_train.sample(5)

In [ ]:
application_train.TARGET.value_counts()

Для удобства далее будем рассматривать лишь 10% данных.


In [ ]:
from sklearn.model_selection import train_test_split

application, _ = train_test_split(..., # Ваш код сюда :)
                                  ..., # Ваш код сюда :)
                                  random_state=27,
                                  stratify=application_train.TARGET # Для сохранения баланса классов
                                  )
application = application.sort_values('SK_ID_CURR').reset_index(drop=True)
application.TARGET.value_counts()

In [ ]:
application.head()

application_train довольно большая таблица, дальше она нам не нужна, можно ее удалить и собрать мусор.

In [ ]:
del application_train

gc.collect(); # Собираем мусор

Выделим числовые и нечисловые признаки.

In [ ]:
categorical_list = []
numerical_list = []
for i in application.columns.tolist():
    if application[i].dtype=='object':
        categorical_list.append(i)
    else:
        numerical_list.append(i)

print('Number of categorical features:', len(categorical_list))
print('Number of numerical features:', len(numerical_list))

Посмотрим на наличие пропущенных значений.

In [ ]:
... # Ваш код сюда :)

Для замены пропущенных значений можно выспользоваться `SimpleImputer` из sklearn: данная модель заменяет пропущенные значения (`np.nan`) каким-то образом `strategy` (по умолчанию заменяет средним, но можно и медиану, самым частым значением или указанной в `fill_value` константой).

In [ ]:
from sklearn.impute import SimpleImputer

application[numerical_list] = ... # Ваш код сюда :)

In [ ]:
application.isnull().sum()

In [ ]:
application.isnull().sum().any()

Теперь разбираемся с категориальными данными.

In [ ]:
application[categorical_list] = application[categorical_list].fillna('Unknown')

In [ ]:
application = pd.get_dummies(application, drop_first=True)
print(application.shape)

In [ ]:
application.isnull().sum().any()

In [ ]:
application.head()

In [ ]:
application.info()

Теперь выделим таргет (TARGET) и удалим SK_ID_CURR (Вопрос: Почему удаляем данный признак?).

In [ ]:
X = application.drop(['SK_ID_CURR', 'TARGET'], axis=1)
y = application.TARGET
feature_name = X.columns.tolist()

In [ ]:
application['SK_ID_CURR'].nunique()

In [ ]:
X.shape

Теперь есть 230 признаков, будем пробовать выбрать лучшие.

## Одномерные методы

Идея: оценить важность каждого признака по отдельности, выбрать самые важные признаки.

In [ ]:
def feature_selector(X, y, score_function, n_features=100):
    importance_list = []
    feature_names = X.columns.to_numpy()
    # Считаем важность для каждого признака
    for i in feature_names:
        importance_list.append(score_function(X[i], y))
    # Заменяем np.nan на 0
    importance_list = [0 if np.isnan(i) else i for i in importance_list]
    # Выбрали названия признаков с наибольшей важностью
    best_features = feature_names[np.argsort(importance_list)[-n_features:]][::-1]

    return best_features

### Корреляция Пирсона

Идея: подсчитали корреляцию признака $x^j$ и таргета ($R(x^j, y)$), если корреляция большая по модулю, значит признак информативный.


$$R(x, y) = \frac{\sum_{i=1}^n(x_i - \overline{x})(y_i - \overline{y})}{\sqrt{\sum_{i=1}^n(x_i - \overline{x})^2 \sum_{i=1}^n(y_i - \overline{y})^2}}$$



In [ ]:
def pearson_correlation_abs(x, y):
  return np.abs(np.corrcoef(x, y)[0, 1])

In [ ]:
cor_features = feature_selector(X, y, score_function=pearson_correlation_abs)
print(str(len(cor_features)), 'selected features')

In [ ]:
fig, axs = plt.subplots(figsize=(10,20), nrows=4, ncols=2)

for i in range(4):
  sns.boxplot(y=X[cor_features[i]],
               x=y,
               ax=axs[i][0])
  sns.histplot(x=X[cor_features[i]],
               hue=y,
               ax=axs[i][1])

Проблема данного подхода: учитывается только линейная связь

In [ ]:
x_ = np.arange(-100, 101)
y_ = x_ ** 2
np.corrcoef(x_, y_)[0, 1]

### 2. T score

Идея: подсчитали t score признака $x^j$ на основе разделения по таргету таргета ($R(x^j, y)$), если t score большой, значит признак информативный.


$$R(x, y) = \frac{|\mu_1 - \mu_0|}{\sqrt{\frac{\sigma_0^2}{n_0}+\frac{\sigma_1^2}{n_1}}}$$, где

$\mu_i, \sigma_i^2, n_i$ - это среднее, дисперсия и количество объектов для признака $x$ класса $i$ (0 или 1).

Данный метод используется для задачи бинарной классификации (для многоклассовой есть F score).

In [ ]:
def t_score(x, y):
  def calc_stats(x):
    return np.mean(x), np.var(x), len(x)

  mu0, s0, n0 = calc_stats(x[y == 0.0])
  mu1, s1, n1 = calc_stats(x[y == 1.0])
  return np.abs(mu1-mu0) / np.sqrt(s0/n0 + s1/n1)

In [ ]:
tscore_features = feature_selector(X, y, score_function=t_score)
print(str(len(tscore_features)), 'selected features')

In [ ]:
fig, axs = plt.subplots(figsize=(10,20), nrows=4, ncols=2)

for i in range(4):
  sns.boxplot(y=X[tscore_features[i]],
               x=y,
               ax=axs[i][0])
  sns.histplot(x=X[tscore_features[i]],
               hue=y,
               ax=axs[i][1])

Можно проверить, что выбрались разные признаки.

In [ ]:
set(cor_features) == set(tscore_features)

### Еще примеры функций (из skleran)




В sklearn реализованы функции для подсчтеа важности признаков:

* Статистика $χ^2$ `chi2`
* F-статистика `f_classif`, `f_regression`
* Взаимная информация `mutual_info_classif`, `mutual_info_regression`


In [ ]:
from sklearn.feature_selection import  mutual_info_classif

mutual_info_classif(X, y)

In [ ]:
def MI_score(x, y):
  return mutual_info_classif(x.values.reshape(-1, 1), y)[0]

mi_features = feature_selector(X, y, score_function=MI_score)
print(str(len(mi_features)), 'selected features')

fig, axs = plt.subplots(figsize=(10,20), nrows=4, ncols=2)

for i in range(4):
  sns.boxplot(y=X[mi_features[i]],
               x=y,
               ax=axs[i][0])
  sns.histplot(x=X[mi_features[i]],
               hue=y,
               ax=axs[i][1])

Основная проблема одномерных методов - не работают, если целевая переменная зависит от совокупности признаков.

## Методы обертки (Wrapper)

Идея:  оценить поднаборы признков, делая возможным обнаружения возможную взаимосвязь между совокупностью признаков.

*RFE* (Recursive Feature Elimination) - жадный метод отбора признаков. `estimator` обучается на начальном наборе признаков, и важность каждого признака получается либо через атрибут `coef_`, либо через атрибут `feature_importances_` модели, указанной в `estimator`.

Затем `step` наименее важных признаков удаляются. Эту процедура рекурсивно повторяется, пока в конечном итоге не будет достигнуто `n_features_to_select` признаков.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_norm = StandardScaler().fit_transform(X)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe_selector = RFE(estimator=LogisticRegression(), n_features_to_select=100, step=10, verbose=5)
rfe_selector.fit(X_norm, y)

In [ ]:
rfe_support = rfe_selector.get_support() # Получаем маску True/False для признаков
rfe_feature = X.loc[:,rfe_support].columns.tolist()
print(str(len(rfe_feature)), 'selected features')

In [ ]:
rfe_feature

Основная проблема - вычислительно дорого.

## Встроенные методы (Embeded)

Идея `SelectFromModel`: через `estimator` подсчитывается важность признаков. Если важность меньше порогового значения - признак убирается. Пороговое значение задается параметром `threshold` -можно задать числом или указать эвристику: “mean”, “median”, дополнительно можно добавить дробь (“0.1*mean”).


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

embeded_lr_selector = SelectFromModel(estimator=LogisticRegression(penalty="l2"), threshold='1.25*median')
embeded_lr_selector.fit(X_norm, y)

In [ ]:
embeded_lr_support = embeded_lr_selector.get_support()
embeded_lr_feature = X.loc[:,embeded_lr_support].columns.tolist()
print(str(len(embeded_lr_feature)), 'selected features')

### L1 регуляризация

Как мы помним при $L1$ регуляризации веса маловажных признаков обнуляются. Чем больше регуляризация, тем больше весов признаков обнуляются.

In [ ]:
l1_selector = SelectFromModel(estimator=LogisticRegression(penalty="l1", solver='liblinear')) # C=1
l1_selector.fit(X_norm[:3000], y[:3000])

In [ ]:
l1_support = l1_selector.get_support()
l1_feature = X.loc[:,l1_support].columns.tolist()
print(str(len(l1_feature)), 'selected features')

Уменьшение C - больше признаков обнуляются.

In [ ]:
l1_selector = SelectFromModel(estimator=LogisticRegression(penalty="l1", solver='liblinear', C=0.5))
l1_selector.fit(X_norm[:3000], y[:3000])

l1_support = l1_selector.get_support()
l1_feature = X.loc[:,l1_support].columns.tolist()
print(str(len(l1_feature)), 'selected features')

In [ ]:
l1_selector = SelectFromModel(estimator=LogisticRegression(penalty="l1", solver='liblinear', C=0.15))
l1_selector.fit(X_norm[:3000], y[:3000])

l1_support = l1_selector.get_support()
l1_feature = X.loc[:,l1_support].columns.tolist()
print(str(len(l1_feature)), 'selected features')

### Random Forest

Для некоторых моделей важность признаков - это атрибут `coef_` (Вопрос: можете привести примеры?), но у леса такого атрибута нет (Вопрос: почему?)


У леса есть атрибут `feature_importances_` - важность признака подсчитывается как нормализованная сумма уменьшений критерия по всем деревьям, по всем вершинам, где было разбиение по данному признаку.

Уменьшение критерия = $H(X_m) - \frac{|X_l|}{|X_m|} H(X_l) - \frac{|X_r|}{|X_m|} H(X_r)$



In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X, y)

In [ ]:
plot = sns.barplot(y=feature_name,
                   x=clf.feature_importances_,
                   order=np.array(feature_name)[np.argsort(clf.feature_importances_)][::-1]
                   )
plot.figure.set_size_inches(10, 50)

Применим `SelectFromModel` к `RandomForestClassifier`.

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

embeded_rf_selector = SelectFromModel(RandomForestClassifier(n_estimators=100),
                                      threshold='1.25*median')
embeded_rf_selector.fit(X, y)

In [ ]:
embeded_rf_support = embeded_rf_selector.get_support()
embeded_rf_feature = X.loc[:,embeded_rf_support].columns.tolist()
print(str(len(embeded_rf_feature)), 'selected features')

### LGBMClassifier

`SelectFromModel` можно использовать не только с моделями из sklearn, например, можно использовать `LGBMClassifier` (у него тоже есть `feature_importances_`).

In [ ]:
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMClassifier
import re

X_renamed = X.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

lgbc=LGBMClassifier(n_estimators=500,
                    learning_rate=0.05,
                    num_leaves=32,
                    colsample_bytree=0.2,
                    reg_alpha=3,
                    reg_lambda=1,
                    min_split_gain=0.01,
                    min_child_weight=40)

embeded_lgb_selector = SelectFromModel(lgbc, threshold='1.25*median')
embeded_lgb_selector.fit(X_renamed, y)

In [ ]:
embeded_lgb_support = embeded_lgb_selector.get_support()
embeded_lgb_feature = X.loc[:,embeded_lgb_support].columns.tolist()
print(str(len(embeded_lgb_feature)), 'selected features')